In [3]:
# bibliotecas
import pandas as pd             # manipulação dos dados
import numpy as np              # suporte de calculos
import matplotlib.pyplot as plt # visualização grafica
import seaborn as sns           # visualização grafica avançada
import plotly.express as px     # visualização grafica Interativa

## Carregamento dos Dados

Carregamos separado porque o dataset já vem dividido temporalmente.
Train = 2019, Test = 2020. Juntar e re-separar aleatoriamente causaria
data leakage, ou seja, informação do futuro vazando para o treino.

In [21]:
# Carrega dados
df_train = pd.read_csv('../data/raw/fraudTrain.csv') # dados de 2019
df_test = pd.read_csv('../data/raw/fraudTest.csv')   # dados de 2020
# linhas x colunas
print(f'df_train: {df_train.shape}')
print(f'df_test: {df_test.shape}')

df_train: (1296675, 23)
df_test: (555719, 23)


## Reconhecimento

Realizando reconhecimento do dados se precisam modelar ou precisam de limpeza primeiro

In [5]:
# visualização df_train
display(df_train.head())

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [22]:
# confirmar compatibilidade de schema
print(df_train.dtypes.equals(df_test.dtypes))
print(f'Colunas train: {df_train.columns.tolist()}')
print(f'Colunas test:  {df_test.columns.tolist()}')

True
Colunas train: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']
Colunas test:  ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


Com a visuluzação resolvi olhar a fundo aque se refere cada dimensão um das dimensões do dataset
- Unnamed: 0 = indice (pode ser removido)
- trans_date_trans_time = data e hora da transacao
- cc_num = numero do cartao (anonimizado) - identificador
- merchant = nome do comerciante
- category = categoria do comerciante (gas_transport, shopping, etc.)
- amt = valor da transacao em dolares
- first = primeiro nome do titular
- last = sobrenome do titular
- gender = genero (M/F)
- street = endereco do titular
- city = cidade do titular
- state = estado do titular
- zip = CEP do titular - identificador
- lat = latitude do titular
- long = longitude do titular
- city_pop = populacao da cidade
- job = profissao do titular
- dob = data de nascimento
- trans_num = ID unico da transacao
- unix_time = timestamp unix
- merch_lat = latitude do comerciante
- merch_long = longitude do comerciante
- is_fraudint = target — 0 legítima, 1 fraude

In [11]:
# visao geral consolidada conjunto de treino
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

In [23]:
# contagem de nulos no conjunto de treino 
df_train.isnull().sum()

Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64

In [17]:
# contagem de duplicados
print(df_train.duplicated().sum())
print(df_test.duplicated().sum())

0
0


In [24]:
# Valores únicos por coluna — identifica colunas de ID e categóricas
df_train.nunique()

Unnamed: 0               1296675
trans_date_trans_time    1274791
cc_num                       983
merchant                     693
category                      14
amt                        52928
first                        352
last                         481
gender                         2
street                       983
city                         894
state                         51
zip                          970
lat                          968
long                         969
city_pop                     879
job                          494
dob                          968
trans_num                1296675
unix_time                1274823
merch_lat                1247805
merch_long               1275745
is_fraud                       2
dtype: int64

## Distribuição  
com o reconhecimento dos dados realizados visto que não temos dados nulos e nem duplicados, e visualizado o que e cada variável representa, seguirei com a análise do comportamento das variaveis individualmente.
Tem slgo estranho que pode enganar o modelo?

In [26]:
# visuliza colunas númericas relevantes 
colunas_numericas = ['amt','lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'unix_time'] # zip e cc_num são identificadores, não métricas, excluí da análise
# analise das colunas numericas
df_train[colunas_numericas].describe().T

,count,mean,std,min,25%,50%,75%,max
amt,1296675.0,7.035104e+01,1.603160e+02,1.000000e+00,9.650000e+00,4.752000e+01,8.314000e+01,2.894890e+04
lat,1296675.0,3.853762e+01,5.075808e+00,2.002710e+01,3.462050e+01,3.935430e+01,4.194040e+01,6.669330e+01
long,1296675.0,-9.022634e+01,1.375908e+01,-1.656723e+02,-9.679800e+01,-8.747690e+01,-8.015800e+01,-6.795030e+01
city_pop,1296675.0,8.882444e+04,3.019564e+05,2.300000e+01,7.430000e+02,2.456000e+03,2.032800e+04,2.906700e+06
merch_lat,1296675.0,3.853734e+01,5.109788e+00,1.902779e+01,3.473357e+01,3.936568e+01,4.195716e+01,6.751027e+01
merch_long,1296675.0,-9.022646e+01,1.377109e+01,-1.666712e+02,-9.689728e+01,-8.743839e+01,-8.023680e+01,-6.695090e+01
unix_time,1296675.0,1.349244e+09,1.284128e+07,1.325376e+09,1.338751e+09,1.349250e+09,1.359385e+09,1.371817e+09
